In [ ]:
import pandas as pd
import numpy as np
import glob
import scanpy as sc
import pyvips
from tqdm.notebook import tqdm
import sys
sys.path.append('../')
from src.preprocess_utils.preprocess_image import get_low_res_image
import matplotlib.pyplot as plt 
import yaml
import anndata as ad

In [ ]:
with open("config_dataset.yaml", "r") as stream:
    samples = yaml.safe_load(stream)["SAMPLE"]

In [ ]:
with open("image_visium_match.yaml", "r") as stream:
    he_to_visium = yaml.safe_load(stream)
visium_to_he = {v:k for k,v in he_to_visium.items()}

In [ ]:
adatas = []
for sample in tqdm(samples):
    adata = sc.read_10x_h5(f"data/LUNG_VISIUM/ebi_downloads/{sample}-filtered_feature_bc_matrix.h5")
    adata.var_names_make_unique()
    sc.pp.filter_genes(adata, min_counts=1)
    adata.obs.index = [f"{b}_{sample}" for b in adata.obs.index]
    adata = adata.copy()
    adata.obs["batch"] = sample.split("_")[0]
    adatas.append(adata)

adatas = ad.concat(adatas)

In [ ]:
sc.pp.pca(adatas, n_comps=15)
sc.external.pp.harmony_integrate(adatas, key="batch")
sc.pp.neighbors(adatas, use_rep="X_pca_harmony")
sc.tl.leiden(adatas, resolution=0.2)

In [ ]:
adatas.obs.to_csv("data/sample_leiden_cluster.csv")